# Alaga Alerta · GenAI — Pipeline RAG

Execute as células em ordem e **salve o notebook com as saídas** (entregável `notebooks/`).
Requer o Ollama rodando (`ollama serve`) com o modelo baixado.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from rag import BaseConhecimento, LIMIAR_BUSCA
from gerar import CONFIGURACOES, sem_contexto, rag_simples, rag_agente

base = BaseConhecimento()
print('Trechos na base:', len(base.trechos))

C:\Users\gukaw\dev\projetos\alaga\genai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Trechos na base: 14


## 1. Busca semântica (item 03)

In [2]:
for t in base.buscar('meu carro ficou preso na enchente, o que faço?'):
    print(f"{t['score']:.3f}  [{t['id']}] {t['fonte']}")

0.886  [c03] Corpo de Bombeiros — Resgate em enchentes
0.884  [c06] Secretaria de Saúde — Pós-enchente
0.857  [c14] Defesa Civil — Evacuação


In [3]:
testset = json.loads((Path.cwd().parent / 'avaliacao' / 'testset.json').read_text(encoding='utf-8'))
print('recall@1 =', round(base.recall_at_k(testset, 1), 3))
print('recall@3 =', round(base.recall_at_k(testset, 3), 3))
print('limiar de recusa =', LIMIAR_BUSCA)

recall@1 = 1.0


recall@3 = 1.0
limiar de recusa = 0.84


## 2. As 3 configurações numa mesma pergunta (itens 04, 05)

In [4]:
pergunta = 'o que faço se meu carro ficar preso na água?'
for nome, fn in CONFIGURACOES.items():
    print('===', nome, '===')
    print(json.dumps(fn(pergunta, base), ensure_ascii=False, indent=2))

=== sem_contexto ===


{
  "resposta": "Se seu carro ficar preso na água, é importante agir rapidamente e com cuidado para evitar acidentes. Primeiramente, evite tentar sair do carro, pois isso pode aumentar o risco de lesões. Em vez disso, tente localizar a fonte do problema, como uma brecha na porta ou uma abertura na tampa do tanque de combustível. Se possível, use um pedaço de madeira ou outro objeto sólido para fechar a brecha e evitar que água entre no veículo. Se você não se sentir confortável em lidar com a situação, procure ajuda de um profissional, como um mecânico ou um serviço de recuperação de veículos. Além disso, se você estiver em uma área onde há risco de alagamento, é importante estar preparado com um kit de emergência que inclua itens como um painel de emergência, um kit de primeiros socorros e um telefone com bateria extra.",
  "fonte": null,
  "recusa": false
}
=== rag_simples ===


{
  "resposta": "Mantenha a calma, não ligue o motor se ele já estiver submerso e saia do veículo. Se as portas não abrirem por causa da pressão da água, saia pela janela. Priorize sua vida, não os pertences.",
  "fonte": "[c03] (fonte: Corpo de Bombeiros — Resgate em enchentes)",
  "recusa": false
}
=== rag_agente ===


{
  "resposta": "Mantenha a calma, não ligue o motor se ele já estiver submerso e saia do veículo. Se as portas não abrirem por causa da pressão da água, saia pela janela. Priorize sua vida, não os pertences.",
  "fonte": "Corpo de Bombeiros — Resgate em enchentes",
  "recusa": false
}


In [5]:
# Pergunta fora de domínio -> deve recusar ('não consta')
print(json.dumps(rag_agente('quem ganhou o campeonato brasileiro?', base), ensure_ascii=False, indent=2))

{
  "resposta": "Não consta na minha base de conhecimento.",
  "fonte": null,
  "recusa": true
}


## 3. Placar completo (itens 06 e 07)

In [6]:
import subprocess
print(subprocess.run([sys.executable, str(Path.cwd().parent / 'avaliacao' / 'avaliar.py')],
                     capture_output=True, text=True).stdout)


[Busca]  recall@1 = 1.000   recall@3 = 1.000   (limiar de recusa = 0.84)

Avaliando config: sem_contexto ...
  acerto=0.0  recusa_correta=0.0  alucinacao=1.0

Avaliando config: rag_simples ...
  acerto=0.4  recusa_correta=1.0  alucinacao=0.429

Avaliando config: rag_agente ...
  acerto=1.0  recusa_correta=1.0  alucinacao=0.0

=== PLACAR (config | acerto | recusa correta | alucinação) ===
  sem_contexto   0.0      0.0              1.0
  rag_simples    0.4      1.0              0.429
  rag_agente     1.0      1.0              0.0

Salvos: placar.json e erros.json (20 erros p/ o item 07)

